In [1]:
"""
User Overlap Features Extraction (Coordination Detection)
==========================================================
检测跨帖用户重叠，识别有组织的协同群体

输入: reddit_wsb_for_network.csv
输出: user_overlap_features_5min.parquet

核心特征（5列）:
1. cross_thread_overlap - 跨帖用户重叠度（Jaccard相似度）
2. multi_thread_user_ratio - 参与多个帖子的用户比例
3. coordinated_group_size - 核心协同群体大小
4. unique_threads - 独立帖子数量
5. thread_concentration - 帖子集中度（Gini系数）

理论依据:
- 协同操纵: 相同用户群体在多个帖子中活跃
- 有机讨论: 用户分散，各帖独立
- 区分grassroots vs astroturfing
"""

import pandas as pd
import numpy as np
from datetime import datetime
from collections import defaultdict, Counter

# ============================================================
# 参数设置
# ============================================================

TIME_WINDOW = 5  # 分钟
DATA_FILE = 'reddit_wsb_for_network.csv'

# ============================================================
# 核心函数
# ============================================================

def jaccard_similarity(set1, set2):
    """
    计算Jaccard相似度
    J(A,B) = |A ∩ B| / |A ∪ B|
    """
    if len(set1) == 0 and len(set2) == 0:
        return 0
    
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    
    if union == 0:
        return 0
    
    return intersection / union


def gini_coefficient(values):
    """
    计算Gini系数（不平等度）
    0 = 完全平等
    1 = 完全不平等
    """
    if len(values) == 0:
        return 0
    
    values = np.array(sorted(values))
    n = len(values)
    
    if n == 0 or values.sum() == 0:
        return 0
    
    index = np.arange(1, n + 1)
    gini = (2 * np.sum(index * values)) / (n * np.sum(values)) - (n + 1) / n
    
    return gini


def extract_user_overlap_features(window_data):
    """
    从时间窗口数据中提取用户重叠特征
    """
    
    if len(window_data) == 0:
        return {
            'cross_thread_overlap': 0,
            'multi_thread_user_ratio': 0,
            'coordinated_group_size': 0,
            'unique_threads': 0,
            'thread_concentration': 0
        }
    
    # ========== 1. 识别独立帖子（threads） ==========
    # 对于posts: post_id就是thread_id
    # 对于comments: 需要找到它们所属的post
    
    # 构建thread映射
    thread_users = defaultdict(set)
    
    for _, row in window_data.iterrows():
        user_id = row['user_id']
        
        # 确定thread_id
        if row['type'] == 'post':
            thread_id = row['post_id']
        else:  # comment
            # post_id列实际上存的是parent post的ID
            thread_id = row['post_id']
        
        thread_users[thread_id].add(user_id)
    
    unique_threads = len(thread_users)
    
    if unique_threads == 0:
        return {
            'cross_thread_overlap': 0,
            'multi_thread_user_ratio': 0,
            'coordinated_group_size': 0,
            'unique_threads': 0,
            'thread_concentration': 0
        }
    
    # ========== 2. 计算跨帖用户重叠 ==========
    if unique_threads >= 2:
        # 计算所有thread pairs的Jaccard相似度
        thread_ids = list(thread_users.keys())
        overlaps = []
        
        for i in range(len(thread_ids)):
            for j in range(i + 1, len(thread_ids)):
                similarity = jaccard_similarity(
                    thread_users[thread_ids[i]],
                    thread_users[thread_ids[j]]
                )
                overlaps.append(similarity)
        
        avg_overlap = np.mean(overlaps) if overlaps else 0
    else:
        avg_overlap = 0
    
    # ========== 3. 多帖参与用户分析 ==========
    # 统计每个用户参与了多少个thread
    user_thread_counts = defaultdict(int)
    
    for thread_id, users in thread_users.items():
        for user in users:
            user_thread_counts[user] += 1
    
    total_users = len(user_thread_counts)
    
    if total_users > 0:
        # 参与多个thread的用户比例
        multi_thread_users = sum(1 for count in user_thread_counts.values() if count > 1)
        multi_thread_ratio = multi_thread_users / total_users
        
        # 核心协同群体大小（参与2+个thread的用户）
        coordinated_users = [user for user, count in user_thread_counts.items() if count >= 2]
        coordinated_size = len(coordinated_users)
    else:
        multi_thread_ratio = 0
        coordinated_size = 0
    
    # ========== 4. Thread集中度（Gini系数） ==========
    # 计算每个thread的用户数分布
    thread_sizes = [len(users) for users in thread_users.values()]
    thread_gini = gini_coefficient(thread_sizes)
    
    return {
        'cross_thread_overlap': avg_overlap,
        'multi_thread_user_ratio': multi_thread_ratio * 100,  # 转为百分比
        'coordinated_group_size': coordinated_size,
        'unique_threads': unique_threads,
        'thread_concentration': thread_gini
    }


# ============================================================
# 主函数
# ============================================================

def main():
    start_time = datetime.now()
    
    print("\n" + "="*70)
    print("USER OVERLAP FEATURES EXTRACTION (COORDINATION DETECTION)")
    print("="*70)
    
    # ========== Step 1: 加载数据 ==========
    print("\nStep 1: Loading Reddit data...")
    
    try:
        reddit_df = pd.read_csv(DATA_FILE)
        reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'])
        
        print(f"  ✓ Loaded {len(reddit_df):,} items")
        print(f"  Date range: {reddit_df['timestamp'].min()} to {reddit_df['timestamp'].max()}")
        
        # 检查必要的列
        required_cols = ['user_id', 'post_id', 'type']
        missing = [col for col in required_cols if col not in reddit_df.columns]
        
        if missing:
            print(f"  ✗ Missing required columns: {missing}")
            return
        
    except FileNotFoundError:
        print(f"  ✗ {DATA_FILE} not found!")
        return
    
    # ========== Step 2: 创建时间窗口 ==========
    print("\nStep 2: Creating time windows...")
    
    reddit_df['time_window'] = reddit_df['timestamp'].dt.floor(f'{TIME_WINDOW}min')
    time_windows = sorted(reddit_df['time_window'].unique())
    
    print(f"  ✓ Created {len(time_windows):,} time windows")
    
    # ========== Step 3: 逐窗口提取特征 ==========
    print("\nStep 3: Extracting user overlap features...")
    
    features_list = []
    
    for i, window_time in enumerate(time_windows):
        if i % 100 == 0:
            print(f"  Processing window {i+1:,}/{len(time_windows):,}...", end='\r')
        
        # 获取该窗口的数据
        window_data = reddit_df[reddit_df['time_window'] == window_time].copy()
        
        # 提取特征
        window_features = extract_user_overlap_features(window_data)
        window_features['timestamp'] = window_time
        
        features_list.append(window_features)
    
    print(f"\n  ✓ Extracted features for {len(features_list):,} windows")
    
    # 转换为DataFrame
    overlap_df = pd.DataFrame(features_list)
    
    # ========== Step 4: 统计分析 ==========
    print("\nStep 4: Feature statistics...")
    
    # 只看有thread活动的窗口
    active = overlap_df[overlap_df['unique_threads'] > 0]
    
    print(f"\n  Thread statistics:")
    print(f"    Windows with threads: {len(active):,} ({len(active)/len(overlap_df)*100:.1f}%)")
    if len(active) > 0:
        print(f"    Mean threads/window: {active['unique_threads'].mean():.1f}")
        print(f"    Max threads/window: {active['unique_threads'].max()}")
    
    print(f"\n  Cross-thread overlap (Jaccard):")
    multi_thread = active[active['unique_threads'] >= 2]
    if len(multi_thread) > 0:
        print(f"    Mean overlap: {multi_thread['cross_thread_overlap'].mean():.3f}")
        print(f"    Max overlap: {multi_thread['cross_thread_overlap'].max():.3f}")
        print(f"    >0.3 (high overlap): {(multi_thread['cross_thread_overlap'] > 0.3).sum():,} windows")
        print(f"    >0.5 (very high): {(multi_thread['cross_thread_overlap'] > 0.5).sum():,} windows")
    else:
        print(f"    No windows with multiple threads")
    
    print(f"\n  Multi-thread participation:")
    if len(active) > 0:
        print(f"    Mean ratio: {active['multi_thread_user_ratio'].mean():.1f}%")
        print(f"    Max ratio: {active['multi_thread_user_ratio'].max():.1f}%")
        print(f"    >30% (coordinated): {(active['multi_thread_user_ratio'] > 30).sum():,} windows")
        print(f"    >50% (highly coordinated): {(active['multi_thread_user_ratio'] > 50).sum():,} windows")
    
    print(f"\n  Coordinated group size:")
    if len(active) > 0:
        print(f"    Mean size: {active['coordinated_group_size'].mean():.1f} users")
        print(f"    Max size: {active['coordinated_group_size'].max()} users")
        print(f"    Windows with groups >10: {(active['coordinated_group_size'] > 10).sum():,}")
        print(f"    Windows with groups >20: {(active['coordinated_group_size'] > 20).sum():,}")
    
    print(f"\n  Thread concentration (Gini):")
    if len(active) > 0:
        print(f"    Mean: {active['thread_concentration'].mean():.3f}")
        print(f"    >0.5 (concentrated): {(active['thread_concentration'] > 0.5).sum():,} windows")
    
    # ========== Step 5: 保存 ==========
    print("\nStep 5: Saving...")
    
    overlap_df.to_parquet('user_overlap_features_5min.parquet', index=False)
    
    print(f"  ✓ Saved: user_overlap_features_5min.parquet")
    print(f"  Shape: {overlap_df.shape}")
    
    # ========== 总结 ==========
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print("\n" + "="*70)
    print("COMPLETE")
    print("="*70)
    
    print(f"\nTime windows: {len(overlap_df):,}")
    print(f"Feature columns: {len(overlap_df.columns)}")
    
    print(f"\nFeatures extracted:")
    print(f"  ✓ cross_thread_overlap (Jaccard similarity)")
    print(f"  ✓ multi_thread_user_ratio (coordination %)")
    print(f"  ✓ coordinated_group_size (core group)")
    print(f"  ✓ unique_threads (thread diversity)")
    print(f"  ✓ thread_concentration (Gini coefficient)")
    
    print(f"\nCoordination signals detected:")
    if len(active) > 0:
        high_overlap = (multi_thread['cross_thread_overlap'] > 0.3).sum() if len(multi_thread) > 0 else 0
        high_ratio = (active['multi_thread_user_ratio'] > 30).sum()
        large_groups = (active['coordinated_group_size'] > 10).sum()
        
        print(f"  Windows with high overlap (>0.3): {high_overlap:,}")
        print(f"  Windows with high multi-thread ratio (>30%): {high_ratio:,}")
        print(f"  Windows with large coordinated groups (>10): {large_groups:,}")
        
        total_suspicious = len(active[
            ((active['unique_threads'] >= 2) & (active['cross_thread_overlap'] > 0.3)) |
            (active['multi_thread_user_ratio'] > 30) |
            (active['coordinated_group_size'] > 10)
        ])
        
        print(f"\n  Total suspicious windows: {total_suspicious:,} ({total_suspicious/len(overlap_df)*100:.1f}%)")
    
    print(f"\nRuntime: {duration:.1f} seconds")
    
    print("\n✓ User overlap feature extraction complete!")
    print("  (Alignment will be done in merge_all_features.py)")
    
    # ========== 解释 ==========
    print("\n" + "="*70)
    print("INTERPRETATION GUIDE")
    print("="*70)
    
    print("\nCoordination indicators:")
    print("  • High overlap (>0.3): Same users across threads")
    print("  • High multi-thread ratio (>30%): Organized participation")
    print("  • Large coordinated groups (>10): Core manipulation group")
    print("  • High concentration (Gini >0.5): Activity focused on few threads")
    
    print("\nOrganic discussion patterns:")
    print("  • Low overlap (<0.2): Users stay in their threads")
    print("  • Low multi-thread ratio (<20%): Independent participation")
    print("  • Small groups (<5): Casual cross-posting")


if __name__ == "__main__":
    main()


USER OVERLAP FEATURES EXTRACTION (COORDINATION DETECTION)

Step 1: Loading Reddit data...
  ✓ Loaded 1,606,093 items
  Date range: 2019-07-01 04:35:02 to 2021-06-29 23:58:28

Step 2: Creating time windows...
  ✓ Created 77,267 time windows

Step 3: Extracting user overlap features...
  Processing window 77,201/77,267...
  ✓ Extracted features for 77,267 windows

Step 4: Feature statistics...

  Thread statistics:
    Windows with threads: 77,267 (100.0%)
    Mean threads/window: 20.8
    Max threads/window: 1261

  Cross-thread overlap (Jaccard):
    Mean overlap: 0.016
    Max overlap: 1.000
    >0.3 (high overlap): 780 windows
    >0.5 (very high): 314 windows

  Multi-thread participation:
    Mean ratio: 3.1%
    Max ratio: 100.0%
    >30% (coordinated): 1,430 windows
    >50% (highly coordinated): 330 windows

  Coordinated group size:
    Mean size: 0.9 users
    Max size: 137 users
    Windows with groups >10: 1,245
    Windows with groups >20: 339

  Thread concentration (Gini